Imports

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from ipywidgets import interact, IntSlider, FloatSlider, Checkbox

Load images

In [2]:
def loadImage(path, size=(200, 200)):
    img = Image.open(path)
    img = img.resize(size)
    return np.array(img, dtype=np.float64) / 255.0

Radon Transform

In [3]:
def radonTransform(image, theta, n, l):
    height, width = image.shape
    center = height // 2
    projection = np.zeros(n)

    sValues = np.linspace(-l/2, l/2, n)

    rad = np.deg2rad(theta)
    cosT = np.cos(rad)
    sinT = np.sin(rad)

    diag = int(np.sqrt(height**2 + width**2))
    tValues = np.linspace(-diag//2, diag//2, diag)

    for i, s in enumerate(sValues):
        xPts = center + s * cosT - tValues * sinT
        yPts = center + s * sinT - tValues * cosT

        valid = (xPts >= 0) & (xPts < height) & (yPts >= 0) & (yPts < width)
        xId = xPts[valid].astype(int)
        yId = yPts[valid].astype(int)

        projection[i] = np.sum(image[xId, yId])

    return projection

Back Projection

In [4]:
def backprojection(reconstruction, projection, theta, n, l):
    height, width = reconstruction.shape
    center = height // 2

    sValues = np.linspace(-l/2, l/2, n)

    rad = np.deg2rad(theta)
    cosT = np.cos(rad)
    sinT = np.sin(rad)

    x, y = np.meshgrid(np.arange(height) - center, np.arange(width) - center)
    
    sCoords = x * cosT + y * sinT
    
    rowBackprojected = np.interp(sCoords, sValues, projection, left=0, right=0)
    reconstruction += rowBackprojected

Interactive Simulation

In [6]:


# Załaduj dane
img = loadImage("obrazy/Kolo.jpg") # Możesz zamienić na load_medical_image('foto.jpg')

@interact(
    delta_alpha=FloatSlider(min=1, max=10, step=1, value=3, description='Krok Δα:'),
    n=IntSlider(min=30, max=180, step=10, value=90, description='Liczba det.:'),
    l=IntSlider(min=50, max=250, step=10, value=150, description='Rozpiętość l:'),
    progress=IntSlider(min=1, max=180, step=1, value=180, description='Postęp obrotu:'),
    show_iterative=Checkbox(value=True, description='Widok iteracyjny')
)
def run_ct(delta_alpha, n, l, progress, show_iterative):
    # Generowanie listy kątów do wykonania
    full_angles = np.arange(0, 180, delta_alpha)
    current_angles = [a for a in full_angles if a <= (progress if show_iterative else 180)]
    
    # Inicjalizacja wyników
    sinogram = np.zeros((len(full_angles), n))
    reconstruction = np.zeros(img.shape)
    
    # Pętla symulacji
    for i, angle in enumerate(current_angles):
        proj = radonTransform(img, angle, n, l)
        sinogram[i, :] = proj
        backprojection(reconstruction, proj, angle, n, l)
        
    # Wyświetlanie
    fig, ax = plt.subplots(1, 3, figsize=(18, 6))
    ax[0].imshow(img, cmap='gray')
    ax[0].set_title("Obraz Wejściowy")
    
    ax[1].imshow(sinogram, cmap='magma', aspect='auto', 
                 extent=[0, n, full_angles[-1], full_angles[0]])
    ax[1].set_title(f"Sinogram (Postęp: {len(current_angles)} projekcji)")
    ax[1].set_ylabel("Kąt θ")
    
    ax[2].imshow(reconstruction, cmap='gray')
    ax[2].set_title("Rekonstrukcja (Obraz Wyjściowy)")
    plt.tight_layout()
    plt.show()

interactive(children=(FloatSlider(value=3.0, description='Krok Δα:', max=10.0, min=1.0, step=1.0), IntSlider(v…